<a href="https://colab.research.google.com/github/iiiiiiiiice/dr1/blob/main/Diabetic_Retinopathy_Keras_Tuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U keras-tuner
import keras_tuner as kt
print(f'Keras Tuner version: {kt.__version__}')
!rm -rf /content/sample_data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.4 MB/s eta 0:00:00
Keras Tuner version: 1.4.8


In [2]:
from google.colab import drive
import os

# Robust check to avoid errors if already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Google Drive already mounted.')

Mounted at /content/drive


```markdown
# Downloading The Dataset
```

In [11]:
# 确保解压到根目录并确认输出文件夹名
!unzip -o /content/Diabetic_Balanced_Data.zip -d /content/
# 列出当前目录结构以排查嵌套问题
!find /content -maxdepth 3 -type d

Archive:  /content/Diabetic_Balanced_Data.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/Diabetic_Balanced_Data.zip or
        /content/Diabetic_Balanced_Data.zip.zip, and cannot find /content/Diabetic_Balanced_Data.zip.ZIP, period.
/content
/content/.config
/content/.config/logs
/content/.config/logs/2026.05.26
/content/.config/configurations
/content/.ipynb_checkpoints
/content/drive
/content/drive/MyDrive
/content/drive/MyDrive/Colab Notebooks
/content/drive/MyDrive/Google AI Studio
/content/drive/MyDrive/Diabetic_Hypertuner
/content/drive/.shortcut-targets-by-id
/content/drive/.Trash-0
/content/drive/.Trash-0/files
/content/drive/.Trash-0/info
/content/drive/.Encrypted
/content/drive/.Encrypted/MyDrive
/content/drive/

In [12]:
import os
import gc
import cv2
import glob
import random
import numpy as np
import pandas as pd
from os import path
from tqdm import tqdm
import seaborn as sns
import tensorflow as tf
from google.colab import drive
from google.colab import files
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam , SGD , RMSprop
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.resnet_v2 import ResNet50V2
from tensorflow.keras.applications.inception_v3 import InceptionV3

In [13]:
def show_data(path_dataset):
  images_data = glob.glob(path_dataset)
  random.shuffle(images_data)
  plt.figure(figsize=(10,10))
  for i in range(9):
    plt.subplot(3,3,i+1)
    img = plt.imread(images_data[i-1])
    plt.imshow(img)

def plot_data(dataset):
  print('Total Number Of Images {}'.format(len(dataset)))
  img_files = [os.path.basename(name) for name in dataset]
  data_label = [str(name.split('/')[-2]) for name in dataset]
  df = pd.DataFrame({'filename':img_files,'label':data_label})
  sns.countplot(x=df['label'])
  df['label'].value_counts()
  return df

In [14]:
# dataset = glob.glob('/content/content/processed_data/*/*.jpeg')
# df = plot_data(dataset)

In [15]:
classes=['No_Dr','Mild','Moderate','severe','Proliferative DR']
class_dict = {}
for i,label in enumerate(classes):
  class_dict[i]=label
print(class_dict)

# label_1 = glob.glob('/content/processed_data/4/*.jpeg')
# label_1 = list(label_1[10000::])
# print(len(label_1))
# for i in range(len(label_1)):
#   os.remove(label_1[i])

{0: 'No_Dr', 1: 'Mild', 2: 'Moderate', 3: 'severe', 4: 'Proliferative DR'}


# Plot Images In A Directory -> Function

## Image Aug using IMAGAUG

In [16]:
import glob
import pandas as pd
import os

# 采用递归搜索，解决嵌套文件夹路径问题
search_pattern = '/content/**/train/*/*.jpeg'
dataset = glob.glob(search_pattern, recursive=True)
print(f'Total Number Of Images found: {len(dataset)}')

if len(dataset) > 0:
    img_paths = dataset
    img_files = [os.path.basename(name) for name in dataset]
    # 提取倒数第二个目录名作为标签
    data_label = [int(name.split(os.sep)[-2]) for name in dataset]
    df = pd.DataFrame({'abs_path': img_paths, 'filename': img_files, 'label': data_label})
    print("Class Distribution:\n", df['label'].value_counts())
else:
    df = pd.DataFrame(columns=['abs_path', 'filename', 'label'])
    print("Warning: No images found. Check if unzip was successful or the search pattern is correct.")

Total Number Of Images found: 0


In [17]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import gc

# 使用绝对路径过滤少数类，避免拼接路径出错
if not df.empty:
    df_minor = df.loc[~df['label'].isin([0, 1, 2])]
    diabetic_imgs = df_minor['abs_path'].values
    print(f"Found {len(diabetic_imgs)} images for augmentation.")
    if len(diabetic_imgs) > 0:
        np.random.shuffle(diabetic_imgs)
else:
    diabetic_imgs = np.array([])
    print("DataFrame is empty, cannot proceed with augmentation.")

gc.collect()

# 修正 Keras 3 预处理层
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation((0.1, 0.3), fill_mode='nearest'),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
])

DataFrame is empty, cannot proceed with augmentation.


In [1]:
# 1. 强制安装 NumPy 1.x 以兼容 imgaug
!pip install --force-reinstall "numpy<2.0" imgaug

import os
import gc
import warnings
import imageio
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from google.colab.patches import cv2_imshow
from tensorflow.keras.preprocessing import image

# 检查 NumPy 版本，如果是 2.x 则提醒重启
if np.__version__.startswith('2.'):
    raise RuntimeError("NumPy 版本仍为 2.x。请点击『代码执行程序』->『重新启动会话』，然后再次运行此单元格。")

# 此时导入 imgaug 应该不会报错了
from imgaug import augmenters as iaa

warnings.filterwarnings('ignore')

seq = iaa.Sequential([
    iaa.Crop(px=(0, 16)),
    iaa.Fliplr(0.5),
    iaa.Affine(rotate=(-25,25)),
    iaa.LinearContrast(alpha=1.2),
    iaa.GaussianBlur(sigma=1.5)
])

def read_img(filename,shape=(512,512)):
    img = image.load_img(filename,target_size=shape)
    img = image.img_to_array(img)
    return img

def load_batch(img_list,batch=32,count=0):
    imgs = []
    fnames = []
    i = count*batch
    for filename in img_list[i:batch*(count+1)]:
        img = read_img(filename)
        imgs.append(img)
        fnames.append(filename)
    return imgs,fnames

# 检查 diabetic_imgs 是否存在，若为空则重新扫描路径
if 'diabetic_imgs' not in locals() or len(diabetic_imgs) == 0:
    import glob
    # 尝试匹配实际解压后的少数类路径 (1, 3, 4)
    diabetic_imgs = []
    for c in ['1', '3', '4']:
        diabetic_imgs.extend(glob.glob(f'/content/Diabetic_Balanced_Data/train/{c}/*.jpeg'))
    diabetic_imgs = np.array(diabetic_imgs)

if len(diabetic_imgs) > 0:
    nb_batches = len(diabetic_imgs)//32
    for idx in tqdm(range(nb_batches), position=0):
        images, fnames = load_batch(diabetic_imgs, count=idx)
        images_aug = seq(images=images)
        for im, im_aug in enumerate(images_aug):
            name = fnames[im][:-5]+'_aug_'+str(im)+'.jpeg'
            imageio.imwrite(name, im_aug)
    print(f"Successfully augmented images.")
else:
    print("No images found for augmentation in /content/Diabetic_Balanced_Data/train/.")

gc.collect()

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached imgaug-0.4.0-py2.py3-none-any.whl.metadata (1.8 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached matplotlib-3.10.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached scikit_image-0.26.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (15 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached imageio-2.37.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached shapely-2.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.8 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  U

No images found for augmentation in /content/Diabetic_Balanced_Data/train/.


0

***Spliting Dataset***

In [ ]:
!pip install -q split-folders
import splitfolders
import os

input_data = '/content/processed_data'
output_data = '/content/Diabetic_Balanced_Data'

if not os.path.exists(output_data):
    os.makedirs(output_data)

if len(os.listdir(output_data)) == 0:
    splitfolders.ratio(input_data, output=output_data, seed=100, ratio=(.7, .2, .1), group_prefix=None)

In [ ]:
# !zip -r /content/Diabetic_Balanced_Data.zip /content/
!mv /content/Diabetic_Balanced_Data.zip /content/drive/MyDrive/

```markdown
#Model Creation
```

In [ ]:
# 统一路径，移除多余的 /content/content
IMG_WIDTH = 256
IMG_HEIGHT = 256
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT)
test_data = '/content/Diabetic_Balanced_Data/test'
training_data = '/content/Diabetic_Balanced_Data/train'
validation_data = '/content/Diabetic_Balanced_Data/val'

In [ ]:
image_data_generator = tf.keras.preprocessing.image.ImageDataGenerator(
  rescale = 1.0/255.0,
  )

training_datagen = image_data_generator.flow_from_directory(
    training_data,
    target_size=IMG_SHAPE,
    shuffle=True,
)

validation_datagen = image_data_generator.flow_from_directory(
    validation_data,
    target_size=IMG_SHAPE,
    shuffle = True
)

test_datagen = image_data_generator.flow_from_directory(
    test_data,
    target_size=IMG_SHAPE,
    shuffle=True)

# KERAS HYPERTUNER

In [2]:
# !rm -rf /content/drive/MyDrive/Diabetic_Hypertuner

In [3]:
import os
import keras_tuner as kt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.resnet_v2 import ResNet50V2

# 1. 自动检测数据路径 (防止嵌套文件夹导致找不到路径)
possible_paths = [
    '/content/Diabetic_Balanced_Data/train',
    '/content/content/Diabetic_Balanced_Data/train',
    '/content/train'
]

training_data = None
for p in possible_paths:
    if os.path.exists(p):
        training_data = p
        validation_data = p.replace('/train', '/val')
        print(f"Detected data path: {training_data}")
        break

if not training_data:
    print("Current files in /content:", os.listdir('/content'))
    raise FileNotFoundError("无法找到训练集文件夹，请检查解压路径。")

# 2. 确保必要的超参数已定义
IMG_WIDTH, IMG_HEIGHT = 256, 256
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT)
classes = ['No_Dr', 'Mild', 'Moderate', 'severe', 'Proliferative DR']

# 3. 定义数据生成器
image_data_generator = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0/255.0)

training_datagen = image_data_generator.flow_from_directory(
    training_data, target_size=IMG_SHAPE, batch_size=32, class_mode='categorical', shuffle=True)

validation_datagen = image_data_generator.flow_from_directory(
    validation_data, target_size=IMG_SHAPE, batch_size=32, class_mode='categorical', shuffle=True)

# 4. 模型构建函数
def model_builder(hp):
    base_model = ResNet50V2(input_shape=(IMG_WIDTH, IMG_HEIGHT, 3), include_top=False, weights='imagenet')
    for layer in base_model.layers[:45]:
        layer.trainable = True
    x = tf.keras.layers.GlobalMaxPooling2D()(base_model.output)
    x = tf.keras.layers.Flatten()(x)
    hp_units = hp.Int('units', min_value=1300, max_value=1750, step=150)
    x = tf.keras.layers.Dense(hp_units, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    prediction_layer = tf.keras.layers.Dense(5, activation='softmax')(x)

    model = tf.keras.Model(inputs=base_model.input, outputs=prediction_layer)
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# 5. 初始化 Tuner 并开始搜索
tuner = kt.Hyperband(model_builder, objective='val_accuracy', max_epochs=10, factor=5,
                     directory='/content/drive/MyDrive/Diabetic_Hypertuner', project_name='diabetic_parameters')

stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

tuner.search(training_datagen, epochs=5, verbose=1, validation_data=validation_datagen, callbacks=[stop_early])

Current files in /content: ['.config', 'Diabetic_Balanced_Data.zip', '.ipynb_checkpoints', 'drive', 'diabetic.zip']


FileNotFoundError: 无法找到训练集文件夹，请检查解压路径。

In [ ]:
best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Hyperparameters: {best_hps}")

model = tuner.hypermodel.build(best_hps)
history = model.fit(training_datagen, epochs=20, validation_data=validation_datagen)

val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % (best_epoch,))

In [ ]:
!rm -rf /content/drive/MyDrive/Diabetic_Weight.h5

In [ ]:
diab_model.save('/content/drive/MyDrive/diab_model.h5')

In [ ]:
import tensorflow as tf
tf.saved_model.save(diab_model,'/content/drive/MyDrive/Diabetic_Weight.h5')

# Model

In [ ]:
%load_ext tensorboard
from datetime import datetime
import os

In [ ]:
def define_model(n_layers=45,BASE_MODEL='ResNet50V2'):
    if BASE_MODEL =='ResNet50V2':
        # Pre-trained model with ResNet50V2
        base_model = ResNet50V2(input_shape=(IMG_WIDTH,IMG_HEIGHT,3),include_top=False,weights='imagenet')
        for layer in base_model.layers[:n_layers]:
            layer.trainable=True
        head_model = base_model.output
        head_model = tf.keras.layers.GlobalMaxPooling2D()(head_model)
        head_model = tf.keras.layers.Flatten(name="Flatten")(head_model)
        head_model = tf.keras.layers.Dense(1600,activation='relu')(head_model)
        head_model = tf.keras.layers.Dropout(0.2)(head_model)
        prediction_layer = tf.keras.layers.Dense(len(classes), activation='softmax')(head_model)
        model = tf.keras.Model(inputs=base_model.input,outputs=prediction_layer)

    if BASE_MODEL =='InceptionV3':
        base_model = InceptionV3(input_shape=(IMG_WIDTH,IMG_HEIGHT,3),include_top=False,weights='imagenet')
        for layer in base_model.layers[:n_layers]:
            layer.trainable=False

        head_model = base_model.output
        head_model = tf.keras.layers.GlobalMaxPooling2D()(head_model)
        head_model = tf.keras.layers.Flatten(name="Flatten")(head_model)
        head_model = tf.keras.layers.Dense(1024,activation='relu')(head_model)
        head_model = tf.keras.layers.Dropout(0.5)(head_model)
        prediction_layer = tf.keras.layers.Dense(len(classes), activation='softmax')(head_model)
        model = tf.keras.Model(inputs=base_model.input,outputs=prediction_layer)
    return model

# define Model
model= define_model(BASE_MODEL='ResNet50V2')

#Compilation of the model
model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    metrics=['accuracy'])

In [ ]:
# 修正 Keras 3 要求：使用 .weights.h5 后缀
checkpoint_path = "/content/drive/MyDrive/Custom_Weights.weights.h5"

cp_callback = ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    mode='min'
)

learning_rate_reduction = ReduceLROnPlateau(
    monitor='val_accuracy',
    patience=2,
    verbose=1,
    factor=0.3,
    min_lr=0.00001
)

logdir = "logs/scalars/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)

In [ ]:
%tensorboard --logdir logs/scalars
history = model.fit(
    training_datagen,
    epochs=12,
    steps_per_epoch=1000,
    shuffle=True,
    validation_data=validation_datagen,
    callbacks=[cp_callback,learning_rate_reduction,tensorboard_callback])

In [ ]:
import gc
gc.collect()

In [ ]:
import tensorflow as tf
tf.saved_model.save(model,'/content/drive/MyDrive/Diabetic_Weight')

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from os.path import join
import numpy as np
import tensorflow as tf

diab_model = load_model('/content/drive/MyDrive/diab_model.h5')
shape = (256,256)

def decode_img(image_path, shape):
    # Note: original code used 'filename', updated to use 'image_path' for consistency
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=shape)
    img = tf.keras.preprocessing.image.img_to_array(img)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    return img

In [ ]:
import glob
import random
import matplotlib.pyplot as plt

test_img = glob.glob('/content/content/Diabetic_Balanced_Data/test/*/*.jpeg')
img_select = random.randint(0, len(test_img) - 1)

print(f"Selected image: {test_img[img_select]}")
img = plt.imread(test_img[img_select])
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
test_data = glob.glob('/content/content/Diabetic_Balanced_Data/test/*/*.jpeg')
print("Test data ", len(test_data))
img_files = [os.path.basename(name) for name in test_data]
test_label = [name.split('/')[-2] for name in test_data]
test_df = pd.DataFrame({'filename': img_files, 'label': test_label})
test_df.to_csv('test_data.csv')
display(test_df)

In [ ]:
predictions = []
for iter, row in test_df.iterrows():
    # Construct the full path using class label and filename
    filename = join('/content/content/Diabetic_Balanced_Data/test/', join(row.label, row.filename))
    # Pre-process image
    img = decode_img(filename, shape)
    # Get model prediction
    pred = diab_model.predict(img)
    y_classes = np.argmax(pred)
    # Store the raw prediction probabilities
    predictions.append(pred)


In [ ]:
test_df['pred_label'] = predictions
print(f"First prediction raw output: {predictions[0]}")
display(test_df.head())

In [ ]:
y_test = test_df['label'].astype(int)
y_pred = test_df['pred_label']

In [ ]:
y_pred = test_df.apply(lambda row: np.argmax(list(row['pred_label'])), axis=1)
print("True labels (y_test):")
print(y_test.values)

In [ ]:
import itertools
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Generate and print the classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Calculate the confusion matrix
cnf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion matrix calculated.")

In [ ]:
def plot_confusion_matrix(cm, classes, title='Confusion matrix', cmap=plt.cm.Blues):
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(10,10))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

np.set_printoptions(precision=2)

# plot normalized confusion matrix
plot_confusion_matrix(cnf_matrix, classes=classes, title='Normalized confusion matrix')
plt.show()

## Saving the Tuner's Best Model

In [6]:
import tensorflow as tf
import os
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras.applications.resnet_v2 import ResNet50V2

tuner_best_model_save_path = '/content/drive/MyDrive/tuner_best_model.h5'

# Ensure required variables for tuner and model are available
# These values are taken from cells WNIA-z-VebFd and cs05sc2ACf7f
if 'IMG_WIDTH' not in locals():
    IMG_WIDTH = 256
if 'IMG_HEIGHT' not in locals():
    IMG_HEIGHT = 256
if 'classes' not in locals():
    classes = ['No_Dr', 'Mild', 'Moderate', 'severe', 'Proliferative DR']

# Re-define model_builder if not already defined (from XEQga0_xohg8)
if 'model_builder' not in locals():
    def model_builder(hp):
        base_model = ResNet50V2(input_shape=(IMG_WIDTH, IMG_HEIGHT, 3), include_top=False, weights='imagenet')
        for layer in base_model.layers[:45]:
            layer.trainable = True
        x = tf.keras.layers.GlobalMaxPooling2D()(base_model.output)
        x = tf.keras.layers.Flatten()(x)
        hp_units = hp.Int('units', min_value=1300, max_value=1750, step=150)
        x = tf.keras.layers.Dense(hp_units, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.3)(x)
        prediction_layer = tf.keras.layers.Dense(len(classes), activation='softmax')(x)

        model_builder_instance = tf.keras.Model(inputs=base_model.input, outputs=prediction_layer)
        hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
        model_builder_instance.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                    loss='categorical_crossentropy', metrics=['accuracy'])
        return model_builder_instance

# Re-establish tuner and get the best model if they are not in scope
if 'tuner' not in locals():
    print("Attempting to load Keras Tuner from disk.")
    tuner_dir = '/content/drive/MyDrive/Diabetic_Hypertuner'
    project_name = 'diabetic_parameters'
    try:
        # Load the tuner, overwrite=False to load existing results
        tuner = kt.Hyperband(model_builder, objective='val_accuracy', max_epochs=10, factor=5,
                             directory=tuner_dir, project_name=project_name, overwrite=False)
        print("Keras Tuner loaded successfully.")
    except Exception as e:
        print(f"Error loading Keras Tuner: {e}")
        tuner = None # Indicate failure

model = None
if tuner:
    try:
        best_models = tuner.get_best_models(num_models=1)
        if best_models:
            # Get the best trained model directly from the tuner
            # This will rebuild the model and load its best weights found during tuning
            model = best_models[0]
            print("Best model from Keras Tuner retrieved successfully.")
            tf.saved_model.save(model, tuner_best_model_save_path)
            print(f"Best model from Keras Tuner saved to: {tuner_best_model_save_path}")
        else:
            print("No best models found by the tuner. Please ensure Keras Tuner search (cell XEQga0_ohg8) was completed successfully.")
    except Exception as e:
        print(f"Error retrieving or saving the best model from tuner: {e}")
        print("Please ensure Keras Tuner search (cell XEQga0_ohg8) and model training (cell DxhxAVtYOgE-) were completed.")
else:
    print("Error: Keras Tuner object could not be loaded or re-initialized. Cannot save model.")


No best models found by the tuner. Please ensure Keras Tuner search (cell XEQga0_ohg8) was completed successfully.


## Evaluating the Tuner's Best Model

In [7]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os # Added import for os.path.exists

# Ensure required variables for data generators are available
# These values are taken from cells cs05sc2ACf7f and K3JR6z_5GVjQ
if 'IMG_WIDTH' not in locals():
    IMG_WIDTH = 256
if 'IMG_HEIGHT' not in locals():
    IMG_HEIGHT = 256
if 'IMG_SHAPE' not in locals():
    IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT)

# Correctly set test_data path
test_data = '/content/Diabetic_Balanced_Data/test'
if not os.path.exists(test_data):
    print(f"Warning: 'test_data' path not found at {test_data}. Please ensure the dataset is unzipped correctly.")
    test_datagen = None # Indicate failure if path is wrong
else:
    # Ensure test_datagen is available, if not, recreate it
    if 'test_datagen' not in locals() or not isinstance(test_datagen, tf.keras.preprocessing.image.DirectoryIterator):
        print("Re-creating ImageDataGenerator and test_datagen.")
        if 'image_data_generator' not in locals():
            image_data_generator = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0/255.0)
        try:
            test_datagen = image_data_generator.flow_from_directory(
                test_data,
                target_size=IMG_SHAPE,
                shuffle=True
            )
            print(f"Successfully re-created test_datagen with {test_datagen.n} images.")
        except Exception as e:
            print(f"Error re-creating test_datagen: {e}")
            test_datagen = None # Indicate failure

# Ensure 'model' is available (should be from the previous modified cell 13cb9d87)
if 'model' not in locals() or model is None:
    print("Error: 'model' object (from Keras Tuner) not found. Cannot make predictions.")
    predictions_tuner_model = None
    tuner_results_df = None # Ensure df is not created if model is missing
elif test_datagen is None:
    print("Error: 'test_datagen' not found due to incorrect path. Cannot make predictions.")
    predictions_tuner_model = None
    tuner_results_df = None
else:
    print("Making predictions with the tuner's best model...")
    try:
        predictions_tuner_model = model.predict(test_datagen)

        # Get true labels from the test_datagen
        test_labels = test_datagen.classes
        class_indices = test_datagen.class_indices
        idx_to_class = {v: k for k, v in class_indices.items()}
        true_labels_names = [idx_to_class[label] for label in test_labels]

        # Convert predictions to class labels
        predicted_labels_indices = np.argmax(predictions_tuner_model, axis=1)
        predicted_labels_names = [idx_to_class[label_idx] for label_idx in predicted_labels_indices]

        # Create a DataFrame for comparison
        tuner_results_df = pd.DataFrame({
            'True_Label': true_labels_names,
            'Predicted_Label': predicted_labels_names
        })
        display(tuner_results_df.head())
        print("Predictions completed for the tuner's best model.")
    except Exception as e:
        print(f"Error during prediction: {e}")
        tuner_results_df = None


Error: 'model' object (from Keras Tuner) not found. Cannot make predictions.


### Debugging Data Paths

In [8]:
import os

print("Contents of /content:")
!ls -F /content

print("\nContents of /content/Diabetic_Balanced_Data/ (if exists):")
if os.path.exists('/content/Diabetic_Balanced_Data'):
    !ls -F /content/Diabetic_Balanced_Data
else:
    print("Directory /content/Diabetic_Balanced_Data does not exist.")

print("\nContents of /content/Diabetic_Balanced_Data/test/ (if exists):")
if os.path.exists('/content/Diabetic_Balanced_Data/test'):
    !ls -F /content/Diabetic_Balanced_Data/test
else:
    print("Directory /content/Diabetic_Balanced_Data/test does not exist.")

print("\nPlease review the paths. If 'test' is inside another subfolder, we'll need to update the `test_data` variable.")

Contents of /content:
drive/

Contents of /content/Diabetic_Balanced_Data/ (if exists):
Directory /content/Diabetic_Balanced_Data does not exist.

Contents of /content/Diabetic_Balanced_Data/test/ (if exists):
Directory /content/Diabetic_Balanced_Data/test does not exist.

Please review the paths. If 'test' is inside another subfolder, we'll need to update the `test_data` variable.


In [3]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import itertools

if 'tuner_results_df' in locals():
    # Generate and print the classification report
    print("\nClassification Report for Tuner's Best Model:")
    print(classification_report(tuner_results_df['True_Label'], tuner_results_df['Predicted_Label']))

    # Calculate the confusion matrix
    cnf_matrix_tuner = confusion_matrix(tuner_results_df['True_Label'], tuner_results_df['Predicted_Label'])
    print("Confusion matrix calculated for Tuner's Best Model.")

    # Define classes for plotting (ensure they are ordered correctly, e.g., from class_indices)
    sorted_classes = sorted(test_datagen.class_indices.keys(), key=lambda x: test_datagen.class_indices[x])

    # Plot normalized confusion matrix
    def plot_confusion_matrix(cm, classes, title='Confusion matrix', cmap=plt.cm.Blues):
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        plt.figure(figsize=(10,10))
        plt.imshow(cm, interpolation='nearest', cmap=cmap)
        plt.title(title)
        plt.colorbar()
        tick_marks = np.arange(len(classes))
        plt.xticks(tick_marks, classes, rotation=45)
        plt.yticks(tick_marks, classes)

        fmt = '.2f'
        thresh = cm.max() / 2.
        for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            plt.text(j, i, format(cm[i, j], fmt),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

        plt.ylabel('True label')
        plt.xlabel('Predicted label')
        plt.tight_layout()

    np.set_printoptions(precision=2)
    plot_confusion_matrix(cnf_matrix_tuner, classes=sorted_classes, title='Normalized Confusion Matrix for Tuner\'s Best Model')
    plt.show()
else:
    print("Tuner's results DataFrame not found. Cannot generate evaluation metrics.")

Tuner's results DataFrame not found. Cannot generate evaluation metrics.
